# Notebook 58 — Symbolic Meta-Law for Coefficient Transport

Notebook 57 showed that the main structure lives in the **coefficient manifold**, not in hidden state-space coordinates.
This notebook turns that result into explicit, sparse, human-readable equations for coefficient transport.

We keep the shared field template

\[
g(r,c;\beta)=\beta_0+\beta_1 c+\beta_2 r+\beta_3 c^3+\beta_4 r^2+\beta_5 r c^2
\]

and aim to learn a symbolic meta-law of the form

\[
\beta = f(\text{system},\text{task},\text{forcing mode},\text{flow mode},k)
\]

where each coefficient has its own sparse, readable equation.

## Main goals

1. Fit sparse symbolic models for each coefficient.
2. Print coefficient transport equations in readable form.
3. Compare symbolic transport against dense linear/ridge baselines.
4. Validate with coefficient, field, and trajectory errors on harder holdout blocks.
5. Summarize which metadata terms drive which coefficient families.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV, Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed template and per-regime coefficient dataset

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())

## Symbolic feature construction

Keep the feature set small and human-readable:
- one-hot metadata terms
- `k`
- `k^2`
- selected interactions:
  - forcing × flow
  - forcing × k
  - system × forcing

In [ ]:
# Sparse symbolic feature design

def build_symbolic_features(df_in, columns=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)

    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")

    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_sf = pd.get_dummies(sf, prefix="sf")

    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )

    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)

    return X.astype(float)

X_symbolic = build_symbolic_features(coef_df)
display(X_symbolic.head())
print("Number of symbolic features:", X_symbolic.shape[1])

## Sparse symbolic regression per coefficient

In [ ]:
# Fit sparse symbolic models

symbolic_models = {}
symbolic_terms = {}
symbolic_preds = pd.DataFrame(index=coef_df.index)

for coef in COEF_COLS:
    X = build_symbolic_features(coef_df)
    y = coef_df[coef].to_numpy(dtype=float)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    model = LassoCV(cv=min(5, len(coef_df)), random_state=42, max_iter=20000)
    model.fit(Xs, y)

    yhat = model.predict(Xs)

    symbolic_models[coef] = {
        "model": model,
        "scaler": scaler,
        "columns": X.columns.tolist(),
    }
    symbolic_preds[coef] = yhat

    raw_coefs = model.coef_
    cols = X.columns.tolist()

    mask = np.abs(raw_coefs) > 1e-3
    symbolic_terms[coef] = [(cols[i], raw_coefs[i]) for i in range(len(cols)) if mask[i]]

display(symbolic_preds.head())

## Pretty-print symbolic equations

In [ ]:
def format_equation(name, terms, intercept):
    if len(terms) == 0:
        return f"{name} = {intercept:.4f}"
    parts = [f"{name} = {intercept:.4f}"]
    for feat, coef in terms:
        sign = "+" if coef >= 0 else "-"
        parts.append(f"{sign} {abs(coef):.4f}·{feat}")
    return " ".join(parts)

equation_rows = []
for coef in COEF_COLS:
    model = symbolic_models[coef]["model"]
    eq = format_equation(coef, symbolic_terms[coef], model.intercept_)
    equation_rows.append({"coefficient": coef, "equation": eq})
    print(eq)
    print()

equations_df = pd.DataFrame(equation_rows)
display(equations_df)

## Symbolic accuracy on the full regime table

In [ ]:
full_acc_rows = []
for coef in COEF_COLS:
    y = coef_df[coef].to_numpy(dtype=float)
    yhat = symbolic_preds[coef].to_numpy(dtype=float)
    full_acc_rows.append({
        "coefficient": coef,
        "rmse": math.sqrt(mean_squared_error(y, yhat)),
        "n_active_terms": len(symbolic_terms[coef]),
    })

full_acc_df = pd.DataFrame(full_acc_rows)
display(full_acc_df)

In [ ]:
# Term counts and importance overview

term_importance_rows = []
for coef in COEF_COLS:
    for feat, coef_val in symbolic_terms[coef]:
        term_importance_rows.append({
            "coefficient": coef,
            "term": feat,
            "abs_weight": abs(coef_val),
        })

term_importance_df = pd.DataFrame(term_importance_rows)
display(term_importance_df.head(20))

In [ ]:
# Plot active term count per coefficient

plt.figure(figsize=(8, 4))
plt.bar(full_acc_df["coefficient"], full_acc_df["n_active_terms"])
plt.title("Active symbolic terms per coefficient")
plt.ylabel("count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Plot top term magnitudes per coefficient

if len(term_importance_df) > 0:
    top_terms = term_importance_df.sort_values("abs_weight", ascending=False).head(25)
    plt.figure(figsize=(10, 5))
    plt.barh(top_terms["coefficient"] + " | " + top_terms["term"], top_terms["abs_weight"])
    plt.title("Largest symbolic transport terms")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## Dense linear / ridge baselines for comparison

In [ ]:
def fit_dense_baseline_models(df_train, df_test):
    X_train = build_symbolic_features(df_train)
    X_test = build_symbolic_features(df_test, columns=X_train.columns)

    Y_train = df_train[COEF_COLS].to_numpy(dtype=float)

    lin = LinearRegression().fit(X_train, Y_train)
    rid = Ridge(alpha=1.0).fit(X_train, Y_train)

    Yhat_lin = lin.predict(X_test)
    Yhat_rid = rid.predict(X_test)

    return Yhat_lin, Yhat_rid

## Hard-block evaluation

Compare:
- symbolic sparse transport
- dense linear transport
- dense ridge transport

In [ ]:
hard_block_masks = {
    "k_extrapolate_high": coef_df["k"].astype(float) == 7.0,
    "k_extrapolate_low": coef_df["k"].astype(float) == 3.0,
    "system_holdout::entropy": coef_df["system"].astype(str) == "entropy",
    "system_holdout::unevenness": coef_df["system"].astype(str) == "unevenness",
}

rows = []

for block_name, test_mask in hard_block_masks.items():
    train_mask = ~test_mask

    df_train = coef_df.loc[train_mask].reset_index(drop=True)
    df_test = coef_df.loc[test_mask].reset_index(drop=True)

    # dense baselines
    Yhat_lin, Yhat_rid = fit_dense_baseline_models(df_train, df_test)

    # symbolic models refit only on train split
    split_symbolic_models = {}
    X_train_base = build_symbolic_features(df_train)
    X_test_base = build_symbolic_features(df_test, columns=X_train_base.columns)

    for coef in COEF_COLS:
        y_train = df_train[coef].to_numpy(dtype=float)

        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(X_train_base)
        Xte_s = scaler.transform(X_test_base)

        model = LassoCV(cv=min(5, len(df_train)), random_state=42, max_iter=20000)
        model.fit(Xtr_s, y_train)

        split_symbolic_models[coef] = {
            "model": model,
            "scaler": scaler,
            "columns": X_train_base.columns.tolist(),
        }

    Yhat_sym = np.zeros((len(df_test), len(COEF_COLS)), dtype=float)
    for j, coef in enumerate(COEF_COLS):
        Xte_s = split_symbolic_models[coef]["scaler"].transform(X_test_base)
        Yhat_sym[:, j] = split_symbolic_models[coef]["model"].predict(Xte_s)

    for i, rid in enumerate(df_test["regime_id"].tolist()):
        sub = regime_subsets[rid]
        beta_true = df_test.loc[df_test["regime_id"] == rid, COEF_COLS].to_numpy(dtype=float)[0]

        beta_sym = Yhat_sym[i]
        beta_lin = Yhat_lin[i]
        beta_rid = Yhat_rid[i]

        y_emp = sub["predicted_flow"].to_numpy(dtype=float)
        pred_sym = predict_with_beta(sub, beta_sym)
        pred_lin = predict_with_beta(sub, beta_lin)
        pred_rid = predict_with_beta(sub, beta_rid)

        rows.append({
            "block": block_name,
            "regime_id": rid,

            "coef_rmse_symbolic": math.sqrt(mean_squared_error(beta_true, beta_sym)),
            "coef_rmse_linear": math.sqrt(mean_squared_error(beta_true, beta_lin)),
            "coef_rmse_ridge": math.sqrt(mean_squared_error(beta_true, beta_rid)),

            "field_rmse_symbolic": math.sqrt(mean_squared_error(y_emp, pred_sym)),
            "field_rmse_linear": math.sqrt(mean_squared_error(y_emp, pred_lin)),
            "field_rmse_ridge": math.sqrt(mean_squared_error(y_emp, pred_rid)),

            "traj_rmse_symbolic": trajectory_gap(sub, beta_true, beta_sym),
            "traj_rmse_linear": trajectory_gap(sub, beta_true, beta_lin),
            "traj_rmse_ridge": trajectory_gap(sub, beta_true, beta_rid),
        })

compare_df = pd.DataFrame(rows)
display(compare_df.head())

In [ ]:
summary_df = compare_df.groupby("block")[[
    "coef_rmse_symbolic", "coef_rmse_linear", "coef_rmse_ridge",
    "field_rmse_symbolic", "field_rmse_linear", "field_rmse_ridge",
    "traj_rmse_symbolic", "traj_rmse_linear", "traj_rmse_ridge",
]].mean().reset_index()

display(summary_df)

In [ ]:
# Comparison plots on harder blocks

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
x = np.arange(len(summary_df))
w = 0.25

axes[0].bar(x - w, summary_df["coef_rmse_symbolic"], width=w, label="symbolic")
axes[0].bar(x, summary_df["coef_rmse_linear"], width=w, label="linear")
axes[0].bar(x + w, summary_df["coef_rmse_ridge"], width=w, label="ridge")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[0].set_title("Coefficient RMSE")
axes[0].legend()

axes[1].bar(x - w, summary_df["field_rmse_symbolic"], width=w, label="symbolic")
axes[1].bar(x, summary_df["field_rmse_linear"], width=w, label="linear")
axes[1].bar(x + w, summary_df["field_rmse_ridge"], width=w, label="ridge")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[1].set_title("Field RMSE")
axes[1].legend()

axes[2].bar(x - w, summary_df["traj_rmse_symbolic"], width=w, label="symbolic")
axes[2].bar(x, summary_df["traj_rmse_linear"], width=w, label="linear")
axes[2].bar(x + w, summary_df["traj_rmse_ridge"], width=w, label="ridge")
axes[2].set_xticks(x)
axes[2].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[2].set_title("Trajectory RMSE")
axes[2].legend()

plt.tight_layout()
plt.show()

## Symbolic term stability across coefficients

In [ ]:
# Binary term-coefficient presence map

all_terms = sorted(set(term_importance_df["term"].tolist())) if len(term_importance_df) > 0 else []
presence = pd.DataFrame(0, index=COEF_COLS, columns=all_terms)

for coef in COEF_COLS:
    active = [term for term, _ in symbolic_terms[coef]]
    for term in active:
        presence.loc[coef, term] = 1

display(presence)

if presence.shape[1] > 0:
    plt.figure(figsize=(max(8, 0.45 * presence.shape[1]), 4))
    plt.imshow(presence.values, aspect="auto", cmap="Greys")
    plt.yticks(range(len(presence.index)), presence.index)
    plt.xticks(range(len(presence.columns)), presence.columns, rotation=45, ha="right")
    plt.title("Symbolic term presence by coefficient")
    plt.tight_layout()
    plt.show()

## Representative symbolic transport case

In [ ]:
# Pick regime where symbolic beats ridge most on trajectory RMSE

rep_idx = int(np.argmax(compare_df["traj_rmse_ridge"] - compare_df["traj_rmse_symbolic"]))
rep = compare_df.iloc[rep_idx]
regime_id = rep["regime_id"]
block_name = rep["block"]

test_mask = coef_df["regime_id"] == regime_id
train_mask = ~test_mask

df_train = coef_df.loc[train_mask].reset_index(drop=True)
df_test = coef_df.loc[test_mask].reset_index(drop=True)

X_train = build_symbolic_features(df_train)
X_test = build_symbolic_features(df_test, columns=X_train.columns)

# Refit symbolic and ridge on split
split_symbolic_models = {}
for coef in COEF_COLS:
    y_train = df_train[coef].to_numpy(dtype=float)

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(X_train)

    model = LassoCV(cv=min(5, len(df_train)), random_state=42, max_iter=20000)
    model.fit(Xtr_s, y_train)

    split_symbolic_models[coef] = {"model": model, "scaler": scaler, "columns": X_train.columns.tolist()}

ridge = Ridge(alpha=1.0).fit(X_train, df_train[COEF_COLS].to_numpy(dtype=float))

beta_true = df_test[COEF_COLS].to_numpy(dtype=float)[0]
beta_ridge = ridge.predict(X_test)[0]
beta_sym = np.array([
    split_symbolic_models[coef]["model"].predict(
        split_symbolic_models[coef]["scaler"].transform(X_test)
    )[0]
    for coef in COEF_COLS
], dtype=float)

coef_compare = pd.DataFrame({
    "term": COEF_COLS,
    "true": beta_true,
    "symbolic": beta_sym,
    "ridge": beta_ridge,
})
display(coef_compare)

In [ ]:
# Representative trajectory comparison

sub = regime_subsets[regime_id]
cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
rmin, rmax = sub["residual"].min(), sub["residual"].max()
cgrid = np.linspace(cmin, cmax, 45)
r0s = np.linspace(np.quantile(sub["residual"], 0.1), np.quantile(sub["residual"], 0.9), 8)
flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

def integrate_beta(beta, r0):
    r = float(np.clip(r0, rmin, rmax))
    traj = [r]
    for i in range(len(cgrid) - 1):
        c = cgrid[i]
        dc = float(cgrid[i+1] - cgrid[i])
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        g = float(np.clip(x @ beta, -flow_cap, flow_cap))
        r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
        traj.append(r)
    return np.array(traj)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
titles = ["True", "Symbolic", "Ridge"]
betas = [beta_true, beta_sym, beta_ridge]

for ax, title, beta in zip(axes, titles, betas):
    for r0 in r0s:
        ax.plot(cgrid, integrate_beta(beta, r0))
    ax.set_title(title)
    ax.set_xlabel("condition coordinate c")
axes[0].set_ylabel("residual")
fig.suptitle(f"Representative symbolic transport case: {regime_id}", y=1.03)
plt.tight_layout()
plt.show()

## Final verdicts

In [ ]:
def verdict_row(row):
    if row["traj_rmse_symbolic"] <= row["traj_rmse_linear"] and row["traj_rmse_symbolic"] <= row["traj_rmse_ridge"]:
        return "symbolic transport competitive/best"
    if row["traj_rmse_ridge"] <= row["traj_rmse_linear"]:
        return "ridge best"
    return "linear best"

summary_df["verdict"] = summary_df.apply(verdict_row, axis=1)
display(summary_df)

## Working conclusion

Notebook 58 extracts explicit sparse transport equations for the governing-field coefficients.

A strong result is:
- each coefficient depends on only a small number of metadata terms
- symbolic transport matches dense baselines on hard blocks
- coefficient families align with readable metadata effects
- the meta-law can be written as equations, not just a predictor

If that pattern holds on your real exports, the next notebook should be:

**Notebook 59 — differential transport law in k**